In [1]:
from math import sqrt
from pathlib import Path
from time import time

import pandas as pd
import numpy as np
from numpy.ma.core import inner

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm

from variables import ALLCLINICAL06_VARS



### Store TXT files as CSV

In [2]:
input_path = Path("data/input")
output_path = Path("data/output")

In [3]:
for file_path in input_path.iterdir():
    file = pd.read_csv(file_path, sep="|")
    file.to_csv(file_path.with_suffix(".csv"), sep="|", index=False)

### Explore AllClinical06

In [24]:
# Read file
all_clinical_06 = pd.read_csv(input_path / "AllClinical06.csv", sep="|")

array([9000099, 9000296, 9000622, ..., 9999862, 9999865, 9999878],
      shape=(4796,))

### exploring missing Data

In [4]:
print("Missing columns:")
for col in all_clinical_06:
    missing = all_clinical_06[col].isna().sum()
    print(f"{col}: {missing}/{len(all_clinical_06[col])}")

Missing columns:
ID: 0/4796
VERSION: 0/4796
V06BLDRAW2: 4779/4796
V06ILLPWK2: 4778/4796
V06MULTST2: 4783/4796
V06URINOB1: 978/4796
V06PLAQHR1: 1037/4796
V06BLUPMN2: 4784/4796
V06HOURSP2: 4785/4796
V06VCOLL2: 4783/4796
V06ILLPWK1: 992/4796
V06MULTST1: 1004/4796
V06VEIN2: 4783/4796
V06URUPMN2: 4787/4796
V06VCOLL1: 1005/4796
V06HOURSP1: 1022/4796
V06HRSUC1: 1020/4796
V06URNCOLL: 540/4796
V06EXCESS2: 4783/4796
V06BLDRAW1: 996/4796
V06URINHR1: 1004/4796
V06HRSUC2: 4787/4796
V06QOVP1: 1005/4796
V06EXCESS1: 1005/4796
V06BLDCOLL: 540/4796
V06SEAQHR1: 1023/4796
V06VOID1: 550/4796
V06QOVP2: 4783/4796
V06VEIN1: 1004/4796
V06OTHVP1: 1004/4796
V06SEAQHR2: 4785/4796
V06OTHVP2: 4783/4796
V06URINHR2: 4787/4796
V06HEMAT1: 1005/4796
V06LEAKAG2: 4783/4796
V06BLSURD1: 4003/4796
V06BLDHRS1: 1021/4796
V06URSURD1: 3995/4796
V06URUPMN1: 1012/4796
V06HEMAT2: 4783/4796
V06LEAKAG1: 1005/4796
V06BLSURD2: 4793/4796
V06VOID2: 4659/4796
V06PLAQHR2: 4785/4796
V06URINOB2: 4665/4796
V06BLDHRS2: 4784/4796
V06URSURD2: 47

## Create working Dataframe

In [5]:
working_df = pd.DataFrame()

# define columns from all_clinical_06
cols = [
    # basic parameter"ID",
    "ID", "V06AGE", "V06WEIGHT", "V06HEIGHT", "V06BMI",
    # pain parameter
    "V06KOOSKPR", "V06P7RKACV", "V06P7RKRCV", "V06KOOSKPL", "V06P7LKACV", "V06P7LKRCV", "V06KGLRS",
    # sleep parameter
    "V06CESD11", "V06WPLKN3", "V06WPRKN3",
    # accelerometer parameter
    "V06AACNT", "V06AALTMNT", "V06AALTMNF", "V06AALTMNS", "V06AAMDMNT", "V06AAMDMNF", "V06AAMDMNS", "V06AAMVMNT","V06AAMVMNF", "V06AAMVMNS", "V06AAMVBMT", "V06AAMVBMF", "V06AAMVBMS", "V06AAVMNT", "V06AAVMNF","V06AAVMNS", "V06AAVBMT", "V06AAVBMF", "V06AAVBMS", "V06AACSM03", "V06ADHHS8", "V06ADHHSD8", "V06AMPA1", "V06ANVDAYS",]

working_df = all_clinical_06[cols]

### aggregate x-ray dataset for merge

In [6]:
xr_df = pd.read_csv(input_path / "KXR_SQ_BU06.csv", sep="|")

# select Kellgren and Lawrence Score and take max grade per ID and side
xr_df_grade = xr_df[["ID", "SIDE", "V06XRKL"]]
xr_df_grade = xr_df_grade.groupby(["ID", "SIDE"])["V06XRKL"].max().reset_index()


# clean label format from SIDE and V06XRKL (e.g. "1: Right" --> "Right", "2: 2" -> "2")
xr_df_grade["SIDE"] = xr_df_grade["SIDE"].str.extract(r":\s*(\w+)")
xr_df_grade["V06XRKL"] = xr_df_grade["V06XRKL"].str.extract(r":\s*(\d+)").squeeze().astype(float).astype("Int64")

# Pivot from long to wide format -> one row per ID, separate columns for Left and Right
xr_df_wide = xr_df_grade.pivot(index="ID", columns="SIDE", values="V06XRKL")
xr_df_wide.columns = [f"V06XRKL_{col}" for col in xr_df_wide.columns]
xr_df_wide = xr_df_wide.reset_index()

"""
V06XRKL" defines Kellgren and Lawrence grade from 0-4

0 = none (definite absence of x-ray changes of osteoarthritis)
1 = doubtful (doubtful joint space narrowing and possible osteophytic lipping)
2 = minimal (definite osteophytes and possible joint space narrowing)
3 = moderate (moderate multiple osteophytes, definite narrowing of joint space, some sclerosis and possible deformity of bone ends)
4 = severe (large osteophytes, marked narrowing of joint space, severe sclerosis and definite deformity of bone ends)
"""

'\nV06XRKL" defines Kellgren and Lawrence grade from 0-4\n\n0 = none (definite absence of x-ray changes of osteoarthritis)\n1 = doubtful (doubtful joint space narrowing and possible osteophytic lipping)\n2 = minimal (definite osteophytes and possible joint space narrowing)\n3 = moderate (moderate multiple osteophytes, definite narrowing of joint space, some sclerosis and possible deformity of bone ends)\n4 = severe (large osteophytes, marked narrowing of joint space, severe sclerosis and definite deformity of bone ends)\n'

### Aggregate Enrollees for SEX column

In [7]:
enrollees_df = pd.read_csv(input_path / "Enrollees.csv", sep="|")

#clean label format from P02SEX (e.g. "1: Male" --> "Male")
enrollees_df["P02SEX"] = enrollees_df["P02SEX"].str.extract(r":\s*(\w+)")


### merge working_df with x-ray (KL grade) and enrollees (sex)

In [8]:
working_df = (working_df
              .merge(xr_df_wide, on="ID", how="inner")
              .merge(enrollees_df[["ID", "P02SEX"]], on="ID", how="inner")
              )


In [9]:
working_df.to_csv(output_path / "working_df.csv", sep="|", index=False)
print(working_df.shape)
print(working_df.columns.tolist())

(3664, 42)
['ID', 'V06AGE', 'V06WEIGHT', 'V06HEIGHT', 'V06BMI', 'V06KOOSKPR', 'V06P7RKACV', 'V06P7RKRCV', 'V06KOOSKPL', 'V06P7LKACV', 'V06P7LKRCV', 'V06KGLRS', 'V06CESD11', 'V06WPLKN3', 'V06WPRKN3', 'V06AACNT', 'V06AALTMNT', 'V06AALTMNF', 'V06AALTMNS', 'V06AAMDMNT', 'V06AAMDMNF', 'V06AAMDMNS', 'V06AAMVMNT', 'V06AAMVMNF', 'V06AAMVMNS', 'V06AAMVBMT', 'V06AAMVBMF', 'V06AAMVBMS', 'V06AAVMNT', 'V06AAVMNF', 'V06AAVMNS', 'V06AAVBMT', 'V06AAVBMF', 'V06AAVBMS', 'V06AACSM03', 'V06ADHHS8', 'V06ADHHSD8', 'V06AMPA1', 'V06ANVDAYS', 'V06XRKL_Left', 'V06XRKL_Right', 'P02SEX']


# Exploring and defining parameters

## Pain

In [10]:
# explore data

pain = all_clinical_06[[
    "ID",
    "V06KOOSKPR",  # Right knee: KOOS Pain Score, 0-100
    # "V06WOMKPR",   # Right knee: WOMAC Pain Score, 0-20
    "V06CPSKR",  # ICOAP Right knee: Constant Pain Score, 0-100
    "V06IPSKR",  # ICOAP Right knee: Intermittent Pain Score, 0-100
    "V06ICPTSKR",  # ICOAP Right knee: Intermittent and Constant Pain Total Score, 0-100
    "V06P7RKFR",  # Right knee pain: how often, 0-4 (never, monthly, weekly, daily, always)
    "V06P7RKACV",  # Right knee pain: on average, past 7 days, rated on scale of 0-10
    "V06P7RKRCV",  # Right knee pain: severity, past 7 days, rated on scale of 0-10

    "V06KOOSKPL",  # Left knee: KOOS Pain Score (calc)
    # "V06WOMKPL",  # Left knee: WOMAC Pain Score (calc)
    "V06CPSKL",  # ICOAP Left knee: Constant Pain Score (calc)
    "V06IPSKL",  # ICOAP Left knee: Intermittent Pain Score (calc)
    "V06ICPTSKL",  # ICOAP Left knee: Intermittent and Constant Pain Total Score (calc)
    "V06P7LKFR",  # Left knee pain: how often
    "V06P7LKACV",  # Left knee pain: on average, past 7 days, rated on scale of 0-10 (calc)
    "V06P7LKRCV",  # Left knee pain: severity, past 7 days, rated on scale of 0-10 (calc)

    "V06KGLRS",  # Considering all ways knee pain and arthritis affect you, how are you doing today? (0–10 scale)
]]

print("Missing columns:")
for col in pain:
    missing = pain[col].isna().sum()
    print(f"{col}: {missing}/{len(pain[col])}")

pain = pain.set_index("ID")
pain = pain.dropna(how="all")

pain.to_csv(output_path / "pain.csv", sep="|", index=False)

print(pain.corr())


Missing columns:
ID: 0/4796
V06KOOSKPR: 562/4796
V06CPSKR: 888/4796
V06IPSKR: 889/4796
V06ICPTSKR: 892/4796
V06P7RKFR: 590/4796
V06P7RKACV: 591/4796
V06P7RKRCV: 590/4796
V06KOOSKPL: 564/4796
V06CPSKL: 888/4796
V06IPSKL: 892/4796
V06ICPTSKL: 893/4796
V06P7LKFR: 589/4796
V06P7LKACV: 589/4796
V06P7LKRCV: 590/4796
V06KGLRS: 558/4796
            V06KOOSKPR  V06CPSKR  V06IPSKR  V06ICPTSKR  V06P7RKFR  V06P7RKACV  \
V06KOOSKPR    1.000000 -0.557875 -0.697575   -0.811913  -0.812341   -0.831895   
V06CPSKR     -0.557875  1.000000  0.216157    0.636900   0.356842    0.470104   
V06IPSKR     -0.697575  0.216157  1.000000    0.890388   0.634891    0.628094   
V06ICPTSKR   -0.811913  0.636900  0.890388    1.000000   0.667960    0.716490   
V06P7RKFR    -0.812341  0.356842  0.634891    0.667960   1.000000    0.705420   
V06P7RKACV   -0.831895  0.470104  0.628094    0.716490   0.705420    1.000000   
V06P7RKRCV   -0.847435  0.435892  0.703792    0.759700   0.771560    0.870655   
V06KOOSKPL    0.56405

In [11]:
# correlations right knee

right_knee_pain = all_clinical_06[["V06KOOSKPR", "V06P7RKFR", "V06P7RKACV", "V06P7RKRCV", "V06KGLRS"]]

print(right_knee_pain.corr())



            V06KOOSKPR  V06P7RKFR  V06P7RKACV  V06P7RKRCV  V06KGLRS
V06KOOSKPR    1.000000  -0.812341   -0.831895   -0.847435 -0.638868
V06P7RKFR    -0.812341   1.000000    0.705420    0.771560  0.489819
V06P7RKACV   -0.831895   0.705420    1.000000    0.870655  0.624710
V06P7RKRCV   -0.847435   0.771560    0.870655    1.000000  0.591812
V06KGLRS     -0.638868   0.489819    0.624710    0.591812  1.000000


## sleep

In [22]:
# explore sleep parameters, explore missing data, remove columns with > 1000 missing data

sleep = all_clinical_06[["ID", "V06CESD11", "V06WPLKN3", "V06WPRKN3"]]

###
# "V06CESD11",  # FU SAQ:Q31k.CES-D: how often sleep was restless, past week 0-3
# "V06WPLKN3",  # FU INT:*Left knee pain: in bed, last 7 days, 0-4
# "V06WPRKN3",  # FU INT:*Right knee pain: in bed, last 7 days, 0-4
# "V06CPLKN2",  # FU INT:Q53r.ICOAP: Left knee constant pain: how much affected sleep, past 7 days
# "V06CPRKN2",  # FU INT:Q53d.ICOAP: Right knee constant pain: how much affected sleep, past 7 days
# '"V06IPLKN3",  # FU INT:Q53y.ICOAP: Left knee intermittent pain: how much affected sleep, past 7 days
# "V06IPRKN3",  # FU INT:Q53k.ICOAP: Right knee intermittent pain: how much affected sleep, past 7 days

print("Missing columns:")
for col in sleep:
    missing = sleep[col].isna().sum()
    print(f"{col}: {missing}/{len(sleep[col])}")

sleep = sleep.set_index('ID')
sleep = sleep.dropna(how="all")

Missing columns:
ID: 0/4796
V06CESD11: 768/4796
V06WPLKN3: 563/4796
V06WPRKN3: 564/4796


In [42]:
# define sleep score scale 0-10
sleep["max_pain_bed"] = sleep[["V06WPLKN3", "V06WPRKN3"]].max(axis=1)
sleep["sleep_score"] = ((sleep["max_pain_bed"] + sleep["V06CESD11"]) / 7 * 10).round(1)

(4248,)


In [24]:
# merge sleep_score to working_df and save as .csv
working_df = working_df.merge(sleep['sleep_score'], on="ID", how="inner")


In [38]:
print(working_df.head())
print(working_df.shape)

        ID  V06AGE  V06WEIGHT  V06HEIGHT  V06BMI  V06KOOSKPR  V06P7RKACV  \
0  9000099    63.0       81.0     1810.0    24.7        97.2         0.0   
1  9001695    56.0       80.5     1650.0    29.6        75.0         2.0   
2  9001897    76.0       79.5     1775.5    25.2       100.0         0.0   
3  9002116    65.0      114.0     1770.5    36.4        97.2         0.0   
4  9002430    72.0       83.3     1781.0    26.3        47.2         5.0   

   V06P7RKRCV  V06KOOSKPL  V06P7LKACV  ...  P02SEX_y  sleep_score  \
0         0.0        97.2         0.0  ...      Male          2.9   
1         2.0       100.0         0.0  ...    Female          2.9   
2         0.0       100.0         0.0  ...      Male          1.4   
3         2.0        86.1         0.0  ...      Male          1.4   
4         8.0        52.8         5.0  ...      Male          5.7   

   wake_minute_mean  wake_minute_sd  sleep_minute_mean  sleep_minute_sd  \
0        423.600000       36.848881        1412.60000

In [26]:
# next step objective restless sleep from minute data --> no accelerometer data during night

## activity

In [12]:
def compute_wake_sleep_time(df: pd.DataFrame, id_col: str = "ID", day_col: str = "V06PAStudyDay",
                            min_col: str = "V06MinSequence", activity_count_col: str = "V06MINCnt",
                            suspect_col: str = "V06SuspectMinute") -> tuple[pd.DataFrame, pd.DataFrame]:

    """
    computes Wake Time (first active minute) and Sleep Time (last active minute) per ID and day
    Returns:
        daily_df: ID, Day, wake_minute, sleep_minute, wake_time_hhmm, sleep_time_hhmm
        id_agg: ID, aggregated Wake/Sleep mean and sds
    """

    def minute_to_hhmm(m):
        if pd.isna(m):
            return np.nan
        h = int((m-1) // 60)
        mins = int((m-1) % 60)
        return f"{h:02d}:{mins:02d}"

    results = []

    for (id, day), grp in df.groupby([id_col, day_col]):

        grp_filtered = grp[grp[suspect_col] == 0].sort_values(min_col)
        active = grp_filtered[grp_filtered[activity_count_col] > 0][min_col].values

        if len(active) == 0:
            results.append({
                id_col: id,
                day_col: day,
                "weekday": grp["V06PAWeekDay"].iloc[0],
                "wake_minute": np.nan,
                "sleep_minute": np.nan,
            })
            continue

        results.append({
            id_col: id,
            day_col: day,
            "weekday": grp["V06PAWeekDay"].iloc[0],
            "wake_minute": active[0], # first active minute
            "sleep_minute": active[-1], # last active minute
        })

    daily_df = pd.DataFrame(results)

    daily_df["wake_time_hhmm"] = daily_df["wake_minute"].apply(minute_to_hhmm)
    daily_df["sleep_time_hhmm"] = daily_df["sleep_minute"].apply(minute_to_hhmm)
    daily_df["wear_duration_min"] = daily_df["sleep_minute"] - daily_df["wake_minute"]

    id_agg = (
        daily_df.groupby(id_col).agg(
            wake_minute_mean = ("wake_minute", np.mean),
            wake_minute_sd = ("wake_minute", np.std),
            sleep_minute_mean = ("sleep_minute", np.mean),
            sleep_minute_sd = ("sleep_minute", np.std),
            wear_duration_mean = ("wear_duration_min", np.mean),
            n_valid_days = ("wake_minute", "count"),
        )
        .reset_index()
    )

    id_agg["wake_mean_hhmm"] = id_agg["wake_minute_mean"].apply(minute_to_hhmm)
    id_agg["sleep_mean_hhmm"] = id_agg["sleep_minute_mean"].apply(minute_to_hhmm)

    return daily_df, id_agg

In [21]:
Acceldatabymin06 = pd.read_csv(input_path / "Acceldatabymin06.csv", sep="|")
Acceldatabymin06["ID"].unique()

array([9000099, 9001695, 9001897, ..., 9999510, 9999862, 9999878],
      shape=(2011,))

In [19]:
daily_06_df, aggregated_df = compute_wake_sleep_time(Acceldatabymin06)
print(daily_06_df.head())
print(aggregated_df.shape)


        ID  V06PAStudyDay   weekday  wake_minute  sleep_minute wake_time_hhmm  \
0  9000099            414    Friday        389.0        1437.0          06:28   
1  9000099            415  Saturday        418.0        1414.0          06:57   
2  9000099            416    Sunday        392.0        1417.0          06:31   
3  9000099            417    Monday        428.0        1389.0          07:07   
4  9000099            418   Tuesday        491.0        1406.0          08:10   

  sleep_time_hhmm  wear_duration_min  
0           23:56             1048.0  
1           23:33              996.0  
2           23:36             1025.0  
3           23:08              961.0  
4           23:25              915.0  
(2011, 9)


### merge dataframes with working_df

In [15]:
working_df = working_df.merge(aggregated_df, on = "ID", how = "inner")


In [20]:
print(working_df.shape)
print(working_df.columns.tolist())

(1951, 50)
['ID', 'V06AGE', 'V06WEIGHT', 'V06HEIGHT', 'V06BMI', 'V06KOOSKPR', 'V06P7RKACV', 'V06P7RKRCV', 'V06KOOSKPL', 'V06P7LKACV', 'V06P7LKRCV', 'V06KGLRS', 'V06CESD11', 'V06WPLKN3', 'V06WPRKN3', 'V06AACNT', 'V06AALTMNT', 'V06AALTMNF', 'V06AALTMNS', 'V06AAMDMNT', 'V06AAMDMNF', 'V06AAMDMNS', 'V06AAMVMNT', 'V06AAMVMNF', 'V06AAMVMNS', 'V06AAMVBMT', 'V06AAMVBMF', 'V06AAMVBMS', 'V06AAVMNT', 'V06AAVMNF', 'V06AAVMNS', 'V06AAVBMT', 'V06AAVBMF', 'V06AAVBMS', 'V06AACSM03', 'V06ADHHS8', 'V06ADHHSD8', 'V06AMPA1', 'V06ANVDAYS', 'V06XRKL_Left', 'V06XRKL_Right', 'P02SEX', 'wake_minute_mean', 'wake_minute_sd', 'sleep_minute_mean', 'sleep_minute_sd', 'wear_duration_mean', 'n_valid_days', 'wake_mean_hhmm', 'sleep_mean_hhmm']


### correlations

In [32]:
subset = working_df[[
    "sleep_score", "V06KOOSKPR",  # 5.1 FU INT: Right knee: KOOS Pain Score (calc)
    "V06P7RKACV",  # Right knee pain: on average, past 7 days, rated on scale of 0-10 (calc)
    "V06P7RKRCV",  # Right knee pain: severity, past 7 days, rated on scale of 0-10 (calc)
    "V06KGLRS",  # Considering all ways knee pain and arthritis affect you, how are you doing today? (0–10 scale),
]]

corr = subset.corr()
corr_pairs = (
    corr.abs()
    .unstack()
    .sort_values(ascending=False)
)

print(corr_pairs[corr_pairs < 1].head(20))


V06P7RKRCV   V06P7RKACV     0.855220
V06P7RKACV   V06P7RKRCV     0.855220
V06P7RKRCV   V06KOOSKPR     0.842779
V06KOOSKPR   V06P7RKRCV     0.842779
             V06P7RKACV     0.826364
V06P7RKACV   V06KOOSKPR     0.826364
V06KOOSKPR   V06KGLRS       0.647151
V06KGLRS     V06KOOSKPR     0.647151
             V06P7RKACV     0.613660
V06P7RKACV   V06KGLRS       0.613660
V06P7RKRCV   V06KGLRS       0.578823
V06KGLRS     V06P7RKRCV     0.578823
V06KOOSKPR   sleep_score    0.530469
sleep_score  V06KOOSKPR     0.530469
V06KGLRS     sleep_score    0.466520
sleep_score  V06KGLRS       0.466520
             V06P7RKACV     0.415733
V06P7RKACV   sleep_score    0.415733
V06P7RKRCV   sleep_score    0.408679
sleep_score  V06P7RKRCV     0.408679
dtype: float64


# Linear regression

In [33]:
def linear_regression_analysis(df: pd.DataFrame, predictor_columns: list[str], target_column: str) -> None:
    """
    Fits a linear regression and provides some analysis on it.
    :param df: The dataframe, containing both the predictor columns and target column.
    :param predictor_columns: List of column names used to predict the target column.
    :param target_column: Single column name of the target column.
    :return: None
    """
    # Drop rows with missing values
    df = df[predictor_columns + [target_column]].dropna()

    X = df[predictor_columns]
    y = df[target_column]

    # Split in train and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

    # Initialize model
    model = LinearRegression()

    # Fit model to training data
    model.fit(X_train, y_train)
    print("--- Fit ---")
    print(f"Coefficients: {model.coef_}")
    print(f"Intercept: {model.intercept_}\n")

    # Predict on test set and compare to actual y_test
    y_pred = model.predict(X_test)
    print("--- Prediction ---")
    print("R²:", r2_score(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("RMSE:", sqrt(mean_squared_error(y_test, y_pred)))

In [34]:
predictor_cols = [
    "V06AAMVMNT", "V06AAMVBMT", "V06AACNT"
]

target_col = "V06P7RKRCV"

linear_regression_analysis(all_clinical_06, predictor_cols, target_col)


# Fehlende Werte entfernen
# data = all_clinical_06[predictor_cols + [target_col]].dropna()


--- Fit ---
Coefficients: [-2.83448029e-02  7.70290671e-03  2.27318383e-06]
Intercept: 2.0799816542929284

--- Prediction ---
R²: 0.0008503587430108706
MSE: 6.175167109315282
RMSE: 2.4849883519476066


In [35]:
import pandas as pd
import numpy as np
from math import sqrt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats


def claude_linear_regression_analysis(df: pd.DataFrame, predictor_columns: list[str], target_column: str) -> None:
    # Drop rows with missing values
    df = df[predictor_columns + [target_column]].dropna()

    X = df[predictor_columns].values
    y = df[target_column].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    residuals = y_train - y_train_pred

    n, p = X_train.shape
    dof = n - p - 1  # degrees of freedom

    # --- Coefficient significance ---
    # Residual standard error and covariance matrix of coefficients
    rse = np.sqrt(np.sum(residuals ** 2) / dof)
    X_aug = np.hstack([np.ones((n, 1)), X_train])  # add intercept column
    cov_matrix = rse ** 2 * np.linalg.inv(X_aug.T @ X_aug)
    se = np.sqrt(np.diag(cov_matrix))  # standard errors

    coefs = np.concatenate([[model.intercept_], model.coef_])
    t_stats = coefs / se
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=dof)
    ci_low = coefs - stats.t.ppf(0.975, df=dof) * se
    ci_high = coefs + stats.t.ppf(0.975, df=dof) * se

    labels = ["intercept"] + predictor_columns
    coef_df = pd.DataFrame({
        "coefficient": coefs,
        "std_err": se,
        "t_stat": t_stats,
        "p_value": p_values,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "significant": ["✓" if p < 0.05 else "✗" for p in p_values],
    }, index=labels)

    print("=" * 60)
    print("COEFFICIENTS")
    print("=" * 60)
    print(coef_df.to_string(float_format="{:.4f}".format))

    # --- Model-level stats ---
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_train - y_train.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    adj_r2 = 1 - (1 - r2) * (n - 1) / dof
    f_stat = (r2 / p) / ((1 - r2) / dof)
    f_p = stats.f.sf(f_stat, p, dof)

    print("\n" + "=" * 60)
    print("MODEL SUMMARY")
    print("=" * 60)
    print(f"  R²:          {r2:.4f}")
    print(f"  Adjusted R²: {adj_r2:.4f}")
    print(f"  F-statistic: {f_stat:.4f}  (p={f_p:.4g})")
    print(f"  RSE:         {rse:.4f}  on {dof} degrees of freedom")

    # --- Train / test performance ---
    train_rmse = sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)

    print("\n" + "=" * 60)
    print("TRAIN / TEST PERFORMANCE (90/10 split)")
    print("=" * 60)
    print(f"  Train — R²: {r2:.4f}  RMSE: {train_rmse:.4f}")
    print(f"  Test  — R²: {test_r2:.4f}  RMSE: {test_rmse:.4f}")
    if r2 - test_r2 > 0.1:
        print(f"  ⚠ R² gap of {r2 - test_r2:.3f} may indicate overfitting.")

    # --- Residual diagnostics ---
    shapiro_stat, shapiro_p = stats.shapiro(residuals)
    dw = np.sum(np.diff(residuals) ** 2) / ss_res  # Durbin-Watson

    print("\n" + "=" * 60)
    print("RESIDUAL DIAGNOSTICS")
    print("=" * 60)
    print(f"  Durbin-Watson: {dw:.4f}  (2 = no autocorrelation)")
    print(f"  Shapiro-Wilk:  W={shapiro_stat:.4f}  p={shapiro_p:.4f}", end="  ")
    print("✓ approx. normal" if shapiro_p > 0.05 else "⚠ may not be normal")

In [36]:
claude_linear_regression_analysis(all_clinical_06, predictor_cols, target_col)

COEFFICIENTS
            coefficient  std_err  t_stat  p_value  ci_low  ci_high significant
intercept        2.0800   0.1743 11.9320   0.0000  1.7381   2.4219           ✓
V06AAMVMNT      -0.0283   0.0105 -2.6984   0.0070 -0.0489  -0.0077           ✓
V06AAMVBMT       0.0077   0.0094  0.8188   0.4130 -0.0107   0.0262           ✗
V06AACNT         0.0000   0.0000  1.8120   0.0702 -0.0000   0.0000           ✗

MODEL SUMMARY
  R²:          0.0101
  Adjusted R²: 0.0083
  F-statistic: 5.8160  (p=0.0005946)
  RSE:         2.5519  on 1715 degrees of freedom

TRAIN / TEST PERFORMANCE (90/10 split)
  Train — R²: 0.0101  RMSE: 2.5490
  Test  — R²: 0.0009  RMSE: 2.4850

RESIDUAL DIAGNOSTICS
  Durbin-Watson: 1.9540  (2 = no autocorrelation)
  Shapiro-Wilk:  W=0.8495  p=0.0000  ⚠ may not be normal
